# Fine-Tuning Large Language Model dengan QLoRA (4-bit NF4) menggunakan Unsloth

**Nama**    : Antonius Vincent Jung  
**Program** : PGABL — Proyek Generative AI  
**Model**   : `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`  
**Dataset** : `Ichsan2895/alpaca-gpt4-indonesian`  
**Teknik**  : QLoRA (4-bit NF4 + Double Quantization) via Unsloth  
**Target**  : Instruction-following dalam Bahasa Indonesia

---

## Deskripsi Proyek

Notebook ini mendokumentasikan proses fine-tuning Large Language Model (LLM)
menggunakan teknik QLoRA (Quantized Low-Rank Adaptation) dengan konfigurasi
4-bit NF4 dan double quantization. Seluruh proses dirancang agar efisien di
GPU T4 Google Colab (15 GB VRAM).

Pipeline utama:

1. **Dataset** — `Ichsan2895/alpaca-gpt4-indonesian`: sekitar 50 ribu instruksi
   Bahasa Indonesia berkualitas GPT-4.
2. **Base model** — `unsloth/Qwen2.5-3B-Instruct-bnb-4bit`: model 3B parameter
   yang sudah di-pre-quantize oleh Unsloth.
3. **Fine-tuning** — QLoRA via Unsloth (2x lebih cepat, 60-70% lebih hemat VRAM
   dibanding implementasi standar HuggingFace).
4. **Eksperimen** — Dua konfigurasi hyperparameter berbeda dijalankan dan
   dibandingkan berdasarkan validation loss.
5. **Logging** — Weights & Biases (W&B) untuk monitoring training secara real-time.
6. **Deployment** — Model terbaik diunggah ke Hugging Face Hub dalam format
   merged 16-bit sehingga dapat langsung digunakan tanpa library PEFT.

---

## Section 1 — Instalasi Dependency

Instalasi seluruh library yang dibutuhkan:

- **Unsloth** : Framework QLoRA yang dioptimasi, 2x lebih cepat dari standar HuggingFace.
- **TRL** : Transformer Reinforcement Learning — menyediakan `SFTTrainer`.
- **PEFT** : Parameter-Efficient Fine-Tuning.
- **BitsAndBytes** : Engine kuantisasi 4-bit.
- **Datasets** : Loading dan preprocessing dataset dari HuggingFace Hub.
- **Weights & Biases** : Experiment tracking dan visualisasi loss curve.

In [1]:
%%capture
# Instalasi Unsloth — framework QLoRA yang dioptimasi (2x lebih cepat dari standar HF)
!pip install unsloth -q

# Library pendukung training
!pip install trl peft accelerate bitsandbytes -q

# Library data & tracking
!pip install datasets wandb huggingface_hub -q

print('Instalasi selesai.')

## Section 2 — Import Library

Import semua library yang diperlukan dan verifikasi ketersediaan GPU.

In [2]:
import os, gc, sys, json
import torch
import wandb
import numpy as np
from datetime import datetime

from datasets import load_dataset, DatasetDict
from huggingface_hub import login as hf_login
from transformers import BitsAndBytesConfig, TrainerCallback
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig

# Verifikasi GPU
assert torch.cuda.is_available(), 'GPU tidak tersedia! Aktifkan T4 di Runtime > Change runtime type'
gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU   : {gpu_name}')
print(f'VRAM  : {vram_gb:.1f} GB')
print(f'Torch : {torch.__version__}')
print(f'Python: {sys.version.split()[0]}')

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:1427: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU   : Tesla T4
VRAM  : 15.6 GB
Torch : 2.10.0+cu128
Python: 3.12.13


## Section 3 — Autentikasi

Login ke Hugging Face dan Weights & Biases menggunakan Colab Secrets.
API key disimpan di panel kiri Colab (ikon kunci), bukan di-hardcode di kode.

- **HF_TOKEN** : Token akses Hugging Face (diperlukan untuk push model).
- **WANDB_API_KEY** : API key Weights & Biases (diperlukan untuk logging).

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# --- Hugging Face ---
# Tambahkan secret 'HF_TOKEN' di panel kiri Colab (ikon kunci)

hf_token  = secrets.get_secret("HF_TOKEN")
hf_login(token=hf_token)
print('Login HuggingFace: OK')

# --- Weights & Biases ---
# Tambahkan secret 'WANDB_API_KEY' di panel kiri Colab

wandb_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_key)
print('Login W&B         : OK')

# Nama project W&B
WANDB_PROJECT = 'PGABL-QLoRA-Qwen2.5-3B'
HF_USERNAME   = 'antonius-vincent'  # Ganti dengan username HF Anda

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Login HuggingFace: OK


wandb: Currently logged in as: antonius-vincent (mc012d5y2159-telkom-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Login W&B         : OK


## Section 4 — Load Dataset

Memuat dataset `Ichsan2895/alpaca-gpt4-indonesian` dari Hugging Face Hub.
Dataset ini berisi sekitar 50 ribu pasang instruksi-respons dalam Bahasa Indonesia
yang dihasilkan menggunakan GPT-4, diformat dalam standar Alpaca.

In [4]:
DATASET_NAME = 'Ichsan2895/alpaca-gpt4-indonesian'

print(f'Memuat dataset: {DATASET_NAME} ...')
raw_ds = load_dataset(DATASET_NAME, split='train')

print(f'Jumlah sampel  : {len(raw_ds):,}')
print(f'Kolom asli     : {raw_ds.column_names}')

# Dataset ini menggunakan kolom 'input' untuk instruksi dan 'output' untuk respons.
# Kolom 'Unnamed: 0' adalah index CSV yang tidak diperlukan — dihapus.
if 'Unnamed: 0' in raw_ds.column_names:
    raw_ds = raw_ds.remove_columns(['Unnamed: 0'])
    print('Kolom index (Unnamed: 0) dihapus.')

print(f'Kolom final    : {raw_ds.column_names}')
print()
print('Contoh pertama:')
sample = raw_ds[0]
print(f'  input  (instruksi): {sample["input"][:120]}')
print(f'  output (respons)  : {sample["output"][:120]}')


Memuat dataset: Ichsan2895/alpaca-gpt4-indonesian ...


README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah sampel  : 49,969
Kolom asli     : ['Unnamed: 0', 'input', 'output']
Kolom index (Unnamed: 0) dihapus.
Kolom final    : ['input', 'output']

Contoh pertama:
  input  (instruksi): Saranlah slogan untuk kampanye daur ulang

  output (respons)  : 1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk 


## Section 5 — Mapping Dataset ke Format ChatML

Setiap baris dataset diubah ke format ChatML (Chat Markup Language) yang
digunakan oleh Qwen2.5-Instruct. Proses mapping dilakukan menggunakan
metode `.map()` dari library HuggingFace `datasets`.

Format ChatML menggunakan special tokens sebagai pembatas peran:

```
<|im_start|>system
{system_prompt}<|im_end|>
<|im_start|>user
{pertanyaan_pengguna}<|im_end|>
<|im_start|>assistant
{respons_model}<|im_end|>
```

In [5]:
# Format ChatML untuk Qwen2.5-Instruct
# Kolom dataset: 'input' -> pesan user, 'output' -> respons asisten

CHATML_START  = '<|im_start|>'
CHATML_END    = '<|im_end|>'
SYSTEM_PROMPT = (
    'Kamu adalah asisten AI yang cerdas, jujur, dan membantu. '
    'Jawab setiap pertanyaan dan selesaikan tugas dengan tepat '
    'menggunakan Bahasa Indonesia yang baik dan benar.'
)

def format_to_chatml(example):
    """
    Mengubah satu baris dataset ke format ChatML untuk Qwen2.5.

    Kolom dataset ini menggunakan:
      'input'  -> instruksi / pertanyaan dari user
      'output' -> respons yang diharapkan dari asisten

    Struktur ChatML output:
      <|im_start|>system\n{system}<|im_end|>\n
      <|im_start|>user\n{user}<|im_end|>\n
      <|im_start|>assistant\n{assistant}<|im_end|>

    Args:
        example (dict): Baris dataset dengan key 'input' dan 'output'
    Returns:
        dict: {'text': str} berisi teks ChatML lengkap
    """
    user_msg  = example['input'].strip()   if example.get('input')  else ''
    assistant = example['output'].strip()  if example.get('output') else ''

    # Skip baris kosong
    if not user_msg or not assistant:
        return {'text': ''}

    text = (
        f'{CHATML_START}system\n{SYSTEM_PROMPT}{CHATML_END}\n'
        f'{CHATML_START}user\n{user_msg}{CHATML_END}\n'
        f'{CHATML_START}assistant\n{assistant}{CHATML_END}'
    )
    return {'text': text}

# Terapkan mapping ke seluruh dataset menggunakan HuggingFace .map()
print('Memformat dataset ke ChatML...')
formatted_ds = raw_ds.map(
    format_to_chatml,
    remove_columns=raw_ds.column_names,
    batched=False,
    num_proc=1,
    desc='ChatML formatting',
)

# Hapus baris kosong (jika ada input/output yang kosong di dataset asli)
before = len(formatted_ds)
formatted_ds = formatted_ds.filter(lambda x: len(x['text']) > 0)
after = len(formatted_ds)
if before != after:
    print(f'Baris kosong dihapus: {before - after}')

print(f'Selesai: {len(formatted_ds):,} sampel | Kolom: {formatted_ds.column_names}')

Memformat dataset ke ChatML...


ChatML formatting (num_proc=1):   0%|          | 0/49969 [00:00<?, ? examples/s]

Filter:   0%|          | 0/49969 [00:00<?, ? examples/s]

Selesai: 49,969 sampel | Kolom: ['text']


## Section 6 — Contoh Hasil Mapping

Cetak satu contoh hasil mapping lengkap beserta special token ChatML.

In [6]:
# Cetak SATU contoh hasil mapping lengkap beserta special token
print('=' * 70)
print('CONTOH HASIL MAPPING (ChatML Format) — indeks 0')
print('=' * 70)
print(formatted_ds[0]['text'])
print('=' * 70)

# Tampilkan panjang karakter & identifikasi special token
ex = formatted_ds[0]['text']
print(f'\nPanjang teks   : {len(ex)} karakter')
print(f'Jumlah <|im_start|>: {ex.count("<|im_start|>")}')
print(f'Jumlah <|im_end|>  : {ex.count("<|im_end|>")}')
print(f'Roles ditemukan: {[r for r in ["system","user","assistant"] if r in ex]}')

CONTOH HASIL MAPPING (ChatML Format) — indeks 0
<|im_start|>system
Kamu adalah asisten AI yang cerdas, jujur, dan membantu. Jawab setiap pertanyaan dan selesaikan tugas dengan tepat menggunakan Bahasa Indonesia yang baik dan benar.<|im_end|>
<|im_start|>user
Saranlah slogan untuk kampanye daur ulang<|im_end|>
<|im_start|>assistant
1. "Kurangi, gunakan kembali, daur ulang: Bersama untuk masa depan yang lebih hijau."
2. "Daur ulanglah hari ini, untuk masa depan yang lebih baik."
3. "Ubah sampahmu menjadi harta karun - Daur ulang!"
4. "Daur ulang untuk siklus kehidupan."
5. "Simpan sumber daya, daur ulang lebih banyak."<|im_end|>

Panjang teks   : 586 karakter
Jumlah <|im_start|>: 3
Jumlah <|im_end|>  : 3
Roles ditemukan: ['system', 'user', 'assistant']


## Section 7 — Split Train dan Validation

Dataset dibagi menjadi dua bagian:
- **Train set** (90%) : digunakan untuk melatih model.
- **Validation set** (10%) : digunakan untuk mengukur generalisasi model
  setelah setiap 200 steps training.

Dataset dibatasi 10.000 sampel agar training selesai dalam waktu wajar
di GPU T4 tanpa risiko Out of Memory (OOM).

In [7]:
# Split dataset: 90% train, 10% validation
# Gunakan seed tetap agar reproducible
TRAIN_RATIO = 0.90
SEED        = 42
MAX_SAMPLES = 10_000  # Batasi untuk efisiensi GPU T4 (hindari OOM)

# Subsample dulu agar training reasonable pada T4
if len(formatted_ds) > MAX_SAMPLES:
  formatted_ds = formatted_ds.shuffle(seed=SEED).select(range(MAX_SAMPLES))
  print(f'Dataset disubsample ke {MAX_SAMPLES:,} sampel')

split = formatted_ds.train_test_split(
  test_size=1 - TRAIN_RATIO,
  seed=SEED,
  shuffle=True,
)

train_ds = split['train']
val_ds   = split['test']

print(f'Train  : {len(train_ds):,} sampel')
print(f'Val    : {len(val_ds):,} sampel')
print(f'Ratio  : {len(train_ds)/len(formatted_ds)*100:.0f}% / {len(val_ds)/len(formatted_ds)*100:.0f}%')

Dataset disubsample ke 10,000 sampel
Train  : 9,000 sampel
Val    : 1,000 sampel
Ratio  : 90% / 10%


## Section 8 — Load Model

Memuat base model `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` menggunakan
`FastLanguageModel` dari Unsloth. Model ini sudah di-pre-quantize dalam
format 4-bit NF4 sehingga proses download lebih cepat dan langsung
siap untuk fine-tuning.

In [8]:
MODEL_NAME     = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048   # Panjang sequence maksimum; T4-safe
DTYPE          = None   # None = auto-detect (float16 di T4)

print(f'Memuat model : {MODEL_NAME}')
print(f'Max seq len  : {MAX_SEQ_LENGTH}')
print('Ini mungkin membutuhkan beberapa menit...')

model, tokenizer = FastLanguageModel.from_pretrained(
model_name      = MODEL_NAME,
max_seq_length  = MAX_SEQ_LENGTH,
dtype           = DTYPE,
load_in_4bit    = True,   # Aktifkan 4-bit quantization (NF4)
)

print()
print(f'Model loaded : {model.__class__.__name__}')
print(f'Tokenizer    : {tokenizer.__class__.__name__}')
print(f'Vocab size   : {tokenizer.vocab_size:,}')
print(f'Pad token    : {repr(tokenizer.pad_token)}')
print(f'EOS token    : {repr(tokenizer.eos_token)}')

# Tampilkan jumlah parameter
total_params = sum(p.numel() for p in model.parameters())
print(f'Total params : {total_params/1e9:.2f}B')

Memuat model : unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Max seq len  : 2048
Ini mungkin membutuhkan beberapa menit...
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


Model loaded : Qwen2ForCausalLM
Tokenizer    : Qwen2Tokenizer
Vocab size   : 151,643
Pad token    : '<|vision_pad|>'
EOS token    : '<|im_end|>'
Total params : 1.70B


## Section 9 — Konfigurasi QLoRA (4-bit NF4 + Double Quantization)

Menampilkan dan menjelaskan konfigurasi BitsAndBytes yang digunakan
secara internal oleh Unsloth ketika `load_in_4bit=True`.

Komponen utama:
- **NF4 (NormalFloat4)** : Tipe kuantisasi 4-bit yang optimal untuk
  distribusi weight pada LLM.
- **Double Quantization** : Mengkuantisasi konstanta kuantisasi itu sendiri,
  menghemat sekitar 0.37 bit per parameter tambahan.
- **Compute dtype float16** : Kalkulasi gradient dalam float16 yang stabil
  di GPU T4 (tidak mendukung bfloat16).

In [9]:
# Tampilkan konfigurasi BitsAndBytes (QLoRA) yang digunakan Unsloth secara eksplisit
# Unsloth menangani ini secara internal, tapi kita tampilkan untuk transparansi
from transformers import BitsAndBytesConfig
import torch

bnb_config_display = BitsAndBytesConfig(
load_in_4bit              = True,        # Aktifkan 4-bit loading
bnb_4bit_quant_type       = 'nf4',       # NormalFloat4 — optimal untuk distribusi weight normal
bnb_4bit_use_double_quant = True,        # Double quantization: kuantisasi konstanta kuantisasi (~0.37 bit/param lebih hemat)
bnb_4bit_compute_dtype    = torch.float16,  # Compute dalam float16 (T4 tidak support bfloat16)
)

print('Konfigurasi QLoRA (4-bit NF4 + Double Quantization):')
print(f'  load_in_4bit              : {bnb_config_display.load_in_4bit}')
print(f'  bnb_4bit_quant_type       : {bnb_config_display.bnb_4bit_quant_type}')
print(f'  bnb_4bit_use_double_quant : {bnb_config_display.bnb_4bit_use_double_quant}')
print(f'  bnb_4bit_compute_dtype    : {bnb_config_display.bnb_4bit_compute_dtype}')
print()
print('Penjelasan:')
print('  NF4               = NormalFloat4, distribusi optimal untuk LLM weights')
print('  Double Quant      = Kuantisasi pada konstanta kuantisasi, hemat ~0.37 bit/param')
print('  Compute dtype f16 = Kalkulasi gradient dalam float16 (stabil di T4)')
print()
print('Note: Unsloth menggunakan config ini secara internal saat load_in_4bit=True')

# Verifikasi VRAM sesudah model load
vram_used = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\nVRAM terpakai  : {vram_used:.2f} GB / {vram_total:.2f} GB')
print(f'VRAM tersisa   : {vram_total - vram_used:.2f} GB')

Konfigurasi QLoRA (4-bit NF4 + Double Quantization):
  load_in_4bit              : True
  bnb_4bit_quant_type       : nf4
  bnb_4bit_use_double_quant : True
  bnb_4bit_compute_dtype    : torch.float16

Penjelasan:
  NF4               = NormalFloat4, distribusi optimal untuk LLM weights
  Double Quant      = Kuantisasi pada konstanta kuantisasi, hemat ~0.37 bit/param
  Compute dtype f16 = Kalkulasi gradient dalam float16 (stabil di T4)

Note: Unsloth menggunakan config ini secara internal saat load_in_4bit=True

VRAM terpakai  : 2.10 GB / 15.64 GB
VRAM tersisa   : 13.54 GB


## Section 10 — Tambah LoRA Adapter

Menambahkan adapter LoRA ke layer-layer target model menggunakan
`FastLanguageModel.get_peft_model()`. LoRA membekukan bobot asli model
dan hanya melatih dua matriks berdimensi rendah (A dan B) per layer,
sehingga jumlah parameter trainable jauh lebih kecil dari total parameter.

Layer target yang dipilih mencakup semua projection layer pada blok
attention dan MLP, yang diketahui memberikan pengaruh terbesar terhadap
kualitas fine-tuning.

In [10]:
# Tambahkan LoRA adapters menggunakan Unsloth get_peft_model
# LoRA membekukan bobot asli dan hanya melatih matriks A & B berdimensi rendah
# sehingga jumlah parameter trainable << jumlah parameter total.

# Target modules — projection layers pada attention & MLP
LORA_TARGET_MODULES = [
'q_proj',    # Query projection (attention)
'k_proj',    # Key projection (attention)
'v_proj',    # Value projection (attention)
'o_proj',    # Output projection (attention)
'gate_proj', # Gate projection (MLP/SwiGLU)
'up_proj',   # Up projection (MLP)
'down_proj', # Down projection (MLP)
]

# Konfigurasi LoRA Eksperimen 1 (akan diubah untuk Eksperimen 2)
LORA_R       = 8    # Rank — dimensi matriks LoRA
LORA_ALPHA   = 16   # Scaling factor (biasanya 2x rank)
LORA_DROPOUT = 0.05 # Dropout regularization

model = FastLanguageModel.get_peft_model(
model,
r                = LORA_R,
target_modules   = LORA_TARGET_MODULES,
lora_alpha       = LORA_ALPHA,
lora_dropout     = LORA_DROPOUT,
bias             = 'none',       # Tidak melatih bias (lebih efisien)
use_gradient_checkpointing = 'unsloth',  # Unsloth gradient checkpointing (hemat VRAM)
random_state     = 42,
use_rslora       = False,
loftq_config     = None,
)

# Hitung parameter trainable
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Parameter trainable : {trainable:,} ({100*trainable/total:.2f}%)')
print(f'Parameter total     : {total:,}')
print(f'Parameter frozen    : {total-trainable:,}')
print(f'LoRA rank (r)       : {LORA_R}')
print(f'LoRA alpha          : {LORA_ALPHA}')
print(f'Target modules      : {LORA_TARGET_MODULES}')

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.7.3 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Parameter trainable : 14,966,784 (0.87%)
Parameter total     : 1,713,639,424
Parameter frozen    : 1,698,672,640
LoRA rank (r)       : 8
LoRA alpha          : 16
Target modules      : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## Section 11 — Eksperimen Hyperparameter

Dua eksperimen dijalankan secara berurutan dengan konfigurasi yang berbeda.
Model di-reload dari base setiap eksperimen agar hasil tidak saling terpengaruh.
Setiap eksperimen ditrack secara terpisah di Weights & Biases.

| Parameter | Eksperimen 1 (Baseline) | Eksperimen 2 (Higher Rank) |
|-----------|------------------------|---------------------------|
| LoRA rank (r) | 8 | 16 |
| LoRA alpha | 16 | 32 |
| LoRA dropout | 0.05 | 0.0 |
| Learning rate | 2e-4 | 1e-4 |
| Batch size | 2 | 2 |
| Grad accumulation | 4 (eff. batch 8) | 8 (eff. batch 16) |
| LR scheduler | cosine | linear |
| Max steps | 800 | 800 |
| Weight decay | 0.01 | 0.001 |

In [11]:
# Definisi dua eksperimen dengan hyperparameter berbeda
# Setiap eksperimen ditrack secara terpisah di Weights & Biases

EXPERIMENTS = [
{
'name'                    : 'Exp1_Baseline_r8',
'run_name'                : 'exp1-baseline-r8-lr2e4',
# LoRA
'lora_r'                  : 8,
'lora_alpha'              : 16,
'lora_dropout'            : 0.05,
# Training
'learning_rate'           : 2e-4,
'per_device_train_batch_size': 2,
'gradient_accumulation_steps': 4,   # effective batch = 2*4 = 8
'max_steps'               : 800,
'warmup_steps'            : 80,
'lr_scheduler_type'       : 'cosine',
'weight_decay'            : 0.01,
},
{
'name'                    : 'Exp2_HigherRank_r16',
'run_name'                : 'exp2-higher-rank-r16-lr1e4',
# LoRA
'lora_r'                  : 16,
'lora_alpha'              : 32,
'lora_dropout'            : 0.0,
# Training
'learning_rate'           : 1e-4,
'per_device_train_batch_size': 2,
'gradient_accumulation_steps': 8,   # effective batch = 2*8 = 16
'max_steps'               : 800,
'warmup_steps'            : 100,
'lr_scheduler_type'       : 'linear',
'weight_decay'            : 0.001,
},
]

print('Rencana Eksperimen:')
print(f'  Jumlah eksperimen : {len(EXPERIMENTS)}')
for i, exp in enumerate(EXPERIMENTS, 1):
  print(f'\n  Eksperimen {i}: {exp["name"]}')
  print(f'    LoRA r={exp["lora_r"]}, alpha={exp["lora_alpha"]}, dropout={exp["lora_dropout"]}')
  print(f'    LR={exp["learning_rate"]}, batch={exp["per_device_train_batch_size"]}, grad_accum={exp["gradient_accumulation_steps"]}')
  print(f'    Steps={exp["max_steps"]}, warmup={exp["warmup_steps"]}, scheduler={exp["lr_scheduler_type"]}')

# Dict untuk menyimpan hasil setiap eksperimen
experiment_results = {}

# Helper: hitung validation loss
def evaluate_validation_loss(trainer):
    """
    Ambil validation loss terakhir dari log_history training.
    ...
    """
    log_history  = trainer.state.log_history
    eval_entries = [e for e in log_history if 'eval_loss' in e]
    if eval_entries:
        return eval_entries[-1]['eval_loss']
    # Fallback: hapus NotebookProgressCallback lalu evaluate()
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
    metrics = trainer.evaluate()
    return metrics.get('eval_loss', float('inf'))


print('\nKonfigurasi eksperimen siap.')

Rencana Eksperimen:
  Jumlah eksperimen : 2

  Eksperimen 1: Exp1_Baseline_r8
    LoRA r=8, alpha=16, dropout=0.05
    LR=0.0002, batch=2, grad_accum=4
    Steps=800, warmup=80, scheduler=cosine

  Eksperimen 2: Exp2_HigherRank_r16
    LoRA r=16, alpha=32, dropout=0.0
    LR=0.0001, batch=2, grad_accum=8
    Steps=800, warmup=100, scheduler=linear

Konfigurasi eksperimen siap.


### Eksperimen 1 — Baseline (r=8, lr=2e-4, cosine scheduler)

Konfigurasi konservatif sebagai baseline. LoRA rank rendah (r=8) dengan
learning rate relatif tinggi dan cosine annealing scheduler.

In [12]:
exp_cfg = EXPERIMENTS[0]
print(f'=== Mulai {exp_cfg["name"]} ===')
print(f'LoRA r={exp_cfg["lora_r"]} | LR={exp_cfg["learning_rate"]} | Steps={exp_cfg["max_steps"]}')

# 1. Hapus model lama dari VRAM
try:
    del model
    gc.collect()
    torch.cuda.empty_cache()
    print('Model lama dihapus dari VRAM')
except NameError:
    pass

# 2. Load model bersih
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = True,
)

# 3. Tambahkan LoRA sesuai config eksperimen 1
model = FastLanguageModel.get_peft_model(
    model,
    r              = exp_cfg['lora_r'],
    target_modules = LORA_TARGET_MODULES,
    lora_alpha     = exp_cfg['lora_alpha'],
    lora_dropout   = exp_cfg['lora_dropout'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

# 4. Inisialisasi W&B run untuk eksperimen 1
run1 = wandb.init(
    project = WANDB_PROJECT,
    name    = exp_cfg['run_name'],
    config  = exp_cfg,
    tags    = ['qwen2.5', 'qlora', 'nf4', 'indonesian'],
)

# 5. Konfigurasi dan jalankan SFTTrainer
from trl import SFTTrainer, SFTConfig

sft_config_1 = SFTConfig(
    output_dir                  = f'./outputs/{exp_cfg["name"]}',
    run_name                    = exp_cfg['run_name'],
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = 'text',
    per_device_train_batch_size = exp_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = exp_cfg['gradient_accumulation_steps'],
    max_steps                   = exp_cfg['max_steps'],
    warmup_steps                = exp_cfg['warmup_steps'],
    learning_rate               = exp_cfg['learning_rate'],
    lr_scheduler_type           = exp_cfg['lr_scheduler_type'],
    weight_decay                = exp_cfg['weight_decay'],
    optim                       = 'adamw_8bit',
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 50,
    eval_strategy               = 'steps',
    eval_steps                  = 200,
    save_strategy               = 'steps',
    save_steps                  = 400,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'wandb',
    seed                        = 42,
    packing                     = True,
)

trainer1 = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args          = sft_config_1,
)

print(f'Training dimulai | {exp_cfg["max_steps"]} steps | {datetime.now().strftime("%H:%M:%S")}')
train_result1 = trainer1.train()

# 6. Evaluasi validation loss
val_loss1   = evaluate_validation_loss(trainer1)
train_loss1 = train_result1.training_loss

print(f'\n[{exp_cfg["name"]}] Training Loss  : {train_loss1:.4f}')
print(f'[{exp_cfg["name"]}] Validation Loss: {val_loss1:.4f}')

wandb.log({'final_train_loss': train_loss1, 'final_val_loss': val_loss1})
run1.finish()

experiment_results[exp_cfg['name']] = {
    'train_loss': train_loss1,
    'val_loss'  : val_loss1,
    'config'    : exp_cfg,
}

# Simpan checkpoint eksperimen 1
model.save_pretrained('./outputs/exp1_checkpoint')
tokenizer.save_pretrained('./outputs/exp1_checkpoint')
print('Model Exp1 disimpan ke ./outputs/exp1_checkpoint')

=== Mulai Exp1_Baseline_r8 ===
LoRA r=8 | LR=0.0002 | Steps=800
Model lama dihapus dari VRAM
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Trainable params: 14,966,784


wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260720_105508-iiyazt6s
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run exp1-baseline-r8-lr2e4
wandb: ⭐️ View project at https://wandb.ai/mc012d5y2159-telkom-university/PGABL-QLoRA-Qwen2.5-3B
wandb: 🚀 View run at https://wandb.ai/mc012d5y2159-telkom-university/PGABL-QLoRA-Qwen2.5-3B/runs/iiyazt6s
wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/9000 [00:00<?, ? examples/s]

...

Step,Training Loss,Validation Loss
200,1.012728,0.985646
400,0.920770,0.974797
600,0.859973,0.978545
800,0.826514,0.985175


Unsloth: Restored added_tokens_decoder metadata in ./outputs/Exp1_Baseline_r8/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./outputs/Exp1_Baseline_r8/checkpoint-800/tokenizer_config.json.
wandb: updating run metadata



[Exp1_Baseline_r8] Training Loss  : 0.9529
[Exp1_Baseline_r8] Validation Loss: 0.9852


wandb: uploading output.log; uploading wandb-summary.json; uploading config.yaml
wandb: 
wandb: Run history:
wandb:               eval/loss █▁▃█
wandb:            eval/runtime █▁▂▃
wandb: eval/samples_per_second ▁█▇▅
wandb:   eval/steps_per_second ▁█▆▅
wandb:        final_train_loss ▁
wandb:          final_val_loss ▁
wandb:             train/epoch ▁▁▂▂▂▃▃▄▄▄▅▅▆▆▆▇▇████
wandb:       train/global_step ▁▁▂▂▂▃▃▄▄▄▅▅▆▆▆▇▇█████
wandb:         train/grad_norm ▂▁▁▃▃▃▅▅▅▅▇▇█▇██
wandb:     train/learning_rate ▅███▇▇▆▅▄▄▃▂▂▁▁▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:               eval/loss 0.98518
wandb:            eval/runtime 140.9061
wandb: eval/samples_per_second 1.057
wandb:   eval/steps_per_second 0.532
wandb:        final_train_loss 0.95289
wandb:          final_val_loss 0.98518
wandb:              total_flos 2.1655478141902848e+17
wandb:             train/epoch 4.82175
wandb:       train/global_step 800
wandb:         train/grad_norm 0.43711
wandb:           

Model Exp1 disimpan ke ./outputs/exp1_checkpoint


### Eksperimen 2 — Higher Rank (r=16, lr=1e-4, linear scheduler)

Konfigurasi dengan kapasitas LoRA lebih tinggi (r=16) dan learning rate
lebih konservatif. Effective batch size lebih besar (16 vs 8) untuk
estimasi gradient yang lebih stabil.

In [13]:
exp_cfg = EXPERIMENTS[1]
print(f'=== Mulai {exp_cfg["name"]} ===')
print(f'LoRA r={exp_cfg["lora_r"]} | LR={exp_cfg["learning_rate"]} | Steps={exp_cfg["max_steps"]}')

# 1. Kosongkan VRAM
del model
gc.collect()
torch.cuda.empty_cache()
print('VRAM dikosongkan untuk Eksperimen 2')

# 2. Load model bersih
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = True,
)

# 3. LoRA dengan rank lebih tinggi
model = FastLanguageModel.get_peft_model(
    model,
    r              = exp_cfg['lora_r'],
    target_modules = LORA_TARGET_MODULES,
    lora_alpha     = exp_cfg['lora_alpha'],
    lora_dropout   = exp_cfg['lora_dropout'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

# 4. W&B run untuk eksperimen 2
run2 = wandb.init(
    project = WANDB_PROJECT,
    name    = exp_cfg['run_name'],
    config  = exp_cfg,
    tags    = ['qwen2.5', 'qlora', 'nf4', 'indonesian'],
)

# 5. SFTTrainer eksperimen 2
sft_config_2 = SFTConfig(
    output_dir                  = f'./outputs/{exp_cfg["name"]}',
    run_name                    = exp_cfg['run_name'],
    max_seq_length              = MAX_SEQ_LENGTH,
    dataset_text_field          = 'text',
    per_device_train_batch_size = exp_cfg['per_device_train_batch_size'],
    per_device_eval_batch_size  = 2,
    gradient_accumulation_steps = exp_cfg['gradient_accumulation_steps'],
    max_steps                   = exp_cfg['max_steps'],
    warmup_steps                = exp_cfg['warmup_steps'],
    learning_rate               = exp_cfg['learning_rate'],
    lr_scheduler_type           = exp_cfg['lr_scheduler_type'],
    weight_decay                = exp_cfg['weight_decay'],
    optim                       = 'adamw_8bit',
    fp16                        = not is_bfloat16_supported(),
    bf16                        = is_bfloat16_supported(),
    logging_steps               = 50,
    eval_strategy               = 'steps',
    eval_steps                  = 200,
    save_strategy               = 'steps',
    save_steps                  = 400,
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,
    report_to                   = 'wandb',
    seed                        = 42,
    packing                     = True,
)

trainer2 = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = val_ds,
    args          = sft_config_2,
)

print(f'Training dimulai | {exp_cfg["max_steps"]} steps | {datetime.now().strftime("%H:%M:%S")}')
train_result2 = trainer2.train()

# 6. Evaluasi validation loss
val_loss2   = evaluate_validation_loss(trainer2)
train_loss2 = train_result2.training_loss

print(f'\n[{exp_cfg["name"]}] Training Loss  : {train_loss2:.4f}')
print(f'[{exp_cfg["name"]}] Validation Loss: {val_loss2:.4f}')

wandb.log({'final_train_loss': train_loss2, 'final_val_loss': val_loss2})
run2.finish()

experiment_results[exp_cfg['name']] = {
    'train_loss': train_loss2,
    'val_loss'  : val_loss2,
    'config'    : exp_cfg,
}

# Simpan checkpoint eksperimen 2
model.save_pretrained('./outputs/exp2_checkpoint')
tokenizer.save_pretrained('./outputs/exp2_checkpoint')
print('Model Exp2 disimpan ke ./outputs/exp2_checkpoint')

=== Mulai Exp2_HigherRank_r16 ===
LoRA r=16 | LR=0.0001 | Steps=800
VRAM dikosongkan untuk Eksperimen 2
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.7.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Trainable params: 29,933,568


wandb: setting up run px0owitw
wandb: Tracking run with wandb version 0.26.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260720_162058-px0owitw
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run exp2-higher-rank-r16-lr1e4
wandb: ⭐️ View project at https://wandb.ai/mc012d5y2159-telkom-university/PGABL-QLoRA-Qwen2.5-3B
wandb: 🚀 View run at https://wandb.ai/mc012d5y2159-telkom-university/PGABL-QLoRA-Qwen2.5-3B/runs/px0owitw


...

Step,Training Loss,Validation Loss
200,1.029715,0.970817
400,0.997884,0.950124
600,0.958837,0.940833
800,0.924894,0.937771


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


[Exp2_HigherRank_r16] Training Loss  : 1.0229
[Exp2_HigherRank_r16] Validation Loss: 0.9378


wandb: uploading history steps 19-21, summary, console lines 41-44
wandb: 
wandb: Run history:
wandb:               eval/loss █▄▂▁
wandb:            eval/runtime ▁▃█▆
wandb: eval/samples_per_second █▆▁▃
wandb:   eval/steps_per_second █▅▁▃
wandb:        final_train_loss ▁
wandb:          final_val_loss ▁
wandb:             train/epoch ▁▁▂▂▂▃▃▄▄▄▅▅▆▆▆▇▇████
wandb:       train/global_step ▁▁▂▂▂▃▃▄▄▄▅▅▆▆▆▇▇█████
wandb:         train/grad_norm ▇▂▁▂▄▃▃▂▃▄▃▅▇▆▅█
wandb:     train/learning_rate ▄██▇▇▆▆▅▅▄▄▃▃▂▂▁
wandb:                      +1 ...
wandb: 
wandb: Run summary:
wandb:               eval/loss 0.93777
wandb:            eval/runtime 206.0526
wandb: eval/samples_per_second 4.853
wandb:   eval/steps_per_second 2.427
wandb:        final_train_loss 1.02286
wandb:          final_val_loss 0.93777
wandb:              total_flos 8.739132591051571e+16
wandb:             train/epoch 1.42133
wandb:       train/global_step 800
wandb:         train/grad_norm 0.64281
wandb:                      +6 .

Model Exp2 disimpan ke ./outputs/exp2_checkpoint


## Section 12 — Perbandingan Hasil Eksperimen

Membandingkan training loss dan validation loss dari kedua eksperimen.
Model dengan validation loss terendah dipilih sebagai model final.

In [14]:
SEP = '=' * 65
print('\n' + SEP)
print('PERBANDINGAN HASIL DUA EKSPERIMEN')
print(SEP)

COL1, COL2, COL3 = 30, 14, 14
header = 'Eksperimen'.ljust(COL1) + 'Train Loss'.rjust(COL2) + 'Val Loss'.rjust(COL3)
print(header)
print('-' * 65)

best_name = None
best_val  = float('inf')

for name, res in experiment_results.items():
    tl  = res['train_loss']
    vl  = res['val_loss']
    row = name.ljust(COL1) + f'{tl:.4f}'.rjust(COL2) + f'{vl:.4f}'.rjust(COL3)
    print(row)
    if vl < best_val:
        best_val  = vl
        best_name = name

print(SEP)
print('Model terbaik berdasarkan Validation Loss:', best_name)
print('Val Loss terbaik :', round(best_val, 4))

best_idx        = list(experiment_results.keys()).index(best_name) + 1
BEST_CHECKPOINT = f'./outputs/exp{best_idx}_checkpoint'
print('Checkpoint terbaik:', BEST_CHECKPOINT)


PERBANDINGAN HASIL DUA EKSPERIMEN
Eksperimen                        Train Loss      Val Loss
-----------------------------------------------------------------
Exp1_Baseline_r8                      0.9529        0.9852
Exp2_HigherRank_r16                   1.0229        0.9378
Model terbaik berdasarkan Validation Loss: Exp2_HigherRank_r16
Val Loss terbaik : 0.9378
Checkpoint terbaik: ./outputs/exp2_checkpoint


## Section 13 — Simpan Model

Memuat kembali checkpoint terbaik dan menyimpan LoRA adapter ke disk.
LoRA adapter jauh lebih kecil dari model penuh (sekitar 100 MB vs beberapa GB)
karena hanya menyimpan bobot yang dilatih.

In [15]:
# Load model terbaik dari checkpoint
print(f'Loading model terbaik dari: {BEST_CHECKPOINT}')

del model
gc.collect()
torch.cuda.empty_cache()

# Load kembali dengan Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
  model_name     = BEST_CHECKPOINT,
  max_seq_length = MAX_SEQ_LENGTH,
  dtype          = DTYPE,
  load_in_4bit   = True,
)
FastLanguageModel.for_inference(model)  # Aktifkan mode inference (lebih cepat)

# Simpan LoRA adapter (bukan merged) — ringan (~100MB)
LORA_SAVE_PATH = './final_model_lora'
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f'LoRA adapter disimpan ke: {LORA_SAVE_PATH}')

import os
lora_size_mb = sum(
  os.path.getsize(os.path.join(root, f))
  for root, _, files in os.walk(LORA_SAVE_PATH)
  for f in files
) / 1e6
print(f'Ukuran LoRA adapter: {lora_size_mb:.1f} MB')
print('File tersimpan:')
for f in sorted(os.listdir(LORA_SAVE_PATH)):
  print(f'  {f}')

Loading model terbaik dari: ./outputs/exp2_checkpoint
==((====))==  Unsloth 2026.7.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth: Restored added_tokens_decoder metadata in ./final_model_lora/tokenizer_config.json.


LoRA adapter disimpan ke: ./final_model_lora
Ukuran LoRA adapter: 131.2 MB
File tersimpan:
  README.md
  adapter_config.json
  adapter_model.safetensors
  chat_template.jinja
  tokenizer.json
  tokenizer_config.json


## Section 14 — Push ke Hugging Face Hub (merged_16bit)

Menggabungkan LoRA adapter ke base model (merge) dan mengupload hasilnya
ke Hugging Face Hub dalam format float16.

Keunggulan format `merged_16bit`:
- Model dapat langsung digunakan tanpa library PEFT.
- Kompatibel dengan semua framework inference standar.
- Tidak perlu menyimpan base model dan adapter secara terpisah.

In [16]:
# Push model ke Hugging Face Hub dalam format merged_16bit
# merged_16bit = LoRA digabung ke base model dalam float16
# Keunggulan: model bisa langsung dipakai tanpa library PEFT

HF_REPO_ID = f'{HF_USERNAME}/Qwen2.5-3B-Instruct-Indonesian-QLoRA'

print(f'Push model ke: {HF_REPO_ID}')
print('Format: merged_16bit (LoRA digabung dengan base model dalam float16)')

# Merge LoRA ke base model dan push langsung ke HF Hub
model.push_to_hub_merged(
repo_id    = HF_REPO_ID,
tokenizer  = tokenizer,
save_method= 'merged_16bit',  # Merge + simpan dalam float16
token      = hf_token,
)

print()
print(f'Model berhasil di-push ke HuggingFace!')
print(f'URL: https://huggingface.co/{HF_REPO_ID}')
print()
print('Detail push:')
print(f'  Repo      : {HF_REPO_ID}')
print(f'  Format    : merged_16bit')
print(f'  Base model: {MODEL_NAME}')
print(f'  Dataset   : Ichsan2895/alpaca-gpt4-indonesian')

Push model ke: antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA
Format: merged_16bit (LoRA digabung dengan base model dalam float16)


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.
[huggingface_hub.hf_api|WARNING]No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:18<01:18, 78.09s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:01<00:00, 60.65s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA`

Model berhasil di-push ke HuggingFace!
URL: https://huggingface.co/antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA

Detail push:
  Repo      : antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA
  Format    : merged_16bit
  Base model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
  Dataset   : Ichsan2895/alpaca-gpt4-indonesian


## Section 15 — Inference

Menguji model hasil fine-tuning dengan beberapa instruksi Bahasa Indonesia.
Model sudah di-merge, sehingga inference dilakukan langsung tanpa adapter.

In [17]:
# Inference sederhana menggunakan model hasil fine-tuning
print('=== INFERENCE DEMO ===')
print('Model:', HF_REPO_ID)
print()

FastLanguageModel.for_inference(model)

def generate_response(instruction, context='', max_new_tokens=256):
    # Format input ke ChatML dan generate respons
    user_msg = (instruction + '\n\nKonteks:\n' + context) if context else instruction
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


test_cases = [
    {
        'instruction': 'Jelaskan apa itu machine learning dalam Bahasa Indonesia yang mudah dipahami.',
        'context': '',
    },
    {
        'instruction': 'Buatkan saya resep nasi goreng yang enak.',
        'context': 'Bahan: nasi, telur, bawang merah, bawang putih, kecap manis, minyak goreng.',
    },
    {
        'instruction': 'Tuliskan 5 manfaat berolahraga secara teratur.',
        'context': '',
    },
]

for i, tc in enumerate(test_cases, 1):
    print('--- Test', i, '---')
    print('Instruksi:', tc['instruction'])
    if tc['context']:
        print('Konteks  :', tc['context'][:70])
    print()
    response = generate_response(tc['instruction'], tc['context'])
    print('Respons  :')
    print(response)
    print()

print('Inference selesai.')


Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== INFERENCE DEMO ===
Model: antonius-vincent/Qwen2.5-3B-Instruct-Indonesian-QLoRA

--- Test 1 ---
Instruksi: Jelaskan apa itu machine learning dalam Bahasa Indonesia yang mudah dipahami.



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respons  :
Machine learning adalah teknik komputer yang memungkinkan sistem untuk belajar dan meningkatkan kinerjanya dari data tanpa harus ditentukan secara eksplisit oleh manusia. Dalam kata lain, ini adalah pendekatan pembelajaran mesin yang menempatkan kekuatan dan kemampuan manusia pada komputer, membuatnya dapat belajar dan mengembangkan diri sendiri. Ini berarti bahwa algoritma machine learning terus belajar dari data, mengidentifikasi pola dan aturan, dan menghasilkan prediksi atau rekomendasi. Teknik ini digunakan di banyak industri seperti medis, otomotif, keuangan, dan lain-lain untuk memberikan hasil yang lebih akurat, efektif, dan cepat daripada penggunaan metode tradisional.

--- Test 2 ---
Instruksi: Buatkan saya resep nasi goreng yang enak.
Konteks  : Bahan: nasi, telur, bawang merah, bawang putih, kecap manis, minyak go



Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Respons  :
Nasi Goreng Sederhana

Bahan:
- 2 sendok makan minyak zaitun
- 1 siung bawang putih cincang halus
- 3 iris bawang bombay cincang halus
- 4 butir telur
- 2 ons nasi basah dicampur dengan garam dan kaldu ayam secukupnya
- 2 sendok makan kecap manis
- Garam dan merica secukupnya untuk penyesuaian rasa

Instruksi:

1. Panaskan minyak zaitun di atas api sedang dalam wajan besar.
2. Tambahkan bawang putih cincang dan masak selama beberapa menit hingga meleleh dan menjadi lembut.
3. Aduk-buka nasi dan campurkan dengan bawang bombay cincang, aduk-aduk sampai tercampur rata.
4. Masukkan telur ke dalam wajan, biarkan menggoreng selama beberapa menit hingga matang dan diserap oleh nasi.
5. Tuangkan kecap manis ke dalam wajan, ad

--- Test 3 ---
Instruksi: Tuliskan 5 manfaat berolahraga secara teratur.

Respons  :
1. Kesehatan fisik: Berolahraga secara teratur dapat meningkatkan kesehatan tubuh secara keseluruhan melalui peningkatan kekuatan otot, pembentukan massa tulang, peningkatan a